# 07 Execution manifest (dry-run only)

Build rollback-ready manifests from the latest dry-run planner output.

This notebook still makes **zero file changes**. It only separates rows into:
- executable manifest
- keep register
- review queue
- blocked rows
- rollback manifest


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
PLAN_PATH = None  # set explicitly if you want; otherwise latest plan_dry_run_*.parquet is used

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUTS_DIR  =', OUTPUTS_DIR)


In [ ]:
from src.reporting import find_latest_output
from src.executor import ManifestConfig, build_execution_bundle, save_manifest_bundle, manifest_summary

if PLAN_PATH is None:
    PLAN_PATH = find_latest_output(OUTPUTS_DIR, 'plan_dry_run_')

print('PLAN_PATH =', PLAN_PATH)
plan = pd.read_parquet(PLAN_PATH)
print('Rows:', len(plan))
preview_cols = [c for c in ['relative_path', 'planner_action', 'planner_ready', 'planner_needs_user_input', 'planner_target_relative_path'] if c in plan.columns]
display(plan[preview_cols].head(15))


## Manifest settings

Leave these conservative at first. In particular, keep collision blocking enabled.


In [ ]:
EXECUTABLE_ACTIONS = (
    'move_to_policy_folder',
    'move_to_superseded_folder',
    'move_to_duplicate_folder',
    'move_to_deprecated_folder',
)
INCLUDE_KEEP_REGISTER = True
BLOCK_TARGET_COLLISIONS = True
REQUIRE_TARGET_PATH = True
REQUIRE_CHANGED_PATH = True

config = ManifestConfig(
    executable_actions=EXECUTABLE_ACTIONS,
    include_keep_register=INCLUDE_KEEP_REGISTER,
    block_target_collisions=BLOCK_TARGET_COLLISIONS,
    require_target_path=REQUIRE_TARGET_PATH,
    require_changed_path=REQUIRE_CHANGED_PATH,
)
config


In [ ]:
bundle = build_execution_bundle(plan, config=config)
summary = manifest_summary(bundle)
summary


In [ ]:
def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(bundle.executable_manifest, ['relative_path', 'planner_action', 'execution_target_relative_path'])
_show(bundle.keep_register, ['relative_path', 'planner_action', 'keep_status'])
_show(bundle.blocked_manifest, ['relative_path', 'planner_action', 'execution_block_reason'])
_show(bundle.review_queue, ['relative_path', 'planner_action', 'planner_reason'])


In [ ]:
STAMP = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
saved = save_manifest_bundle(bundle, OUTPUTS_DIR, stem=STAMP)
saved
